# drfn-eval2: better decoding + causal slot tests (no training)

Reads `best.pt` from the four earlier notebooks and, for every run:

1. **Re-decodes the full dev set** with beam 5 + no-repeat-4-gram blocking (fixed in
   advance), plus beam 5 without blocking and greedy + blocking. Greedy decoding
   looped in 49-68% of outputs.
2. **Slot swap** (grounded memory runs): force one Ge'ez word's root slot to another
   root and check whether the Amharic output switches to that root's words.
3. **Memory ablation**: switch the memory off and measure the loss increase on
   Amharic tokens linked to rare / unseen Ge'ez word forms.

In [ ]:
!pip install -q tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.device_count())

In [ ]:
import glob, os, shutil, subprocess
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_eval2.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
cks = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
print(len(cks), 'checkpoints'); print('\n'.join(cks))

In [ ]:
print('exit', subprocess.call(['bash', 'run_eval2.sh']))

In [ ]:
import json
rows = []
for p in sorted(glob.glob('/kaggle/working/eval2/*/eval2.json')):
    r = json.load(open(p)); d = r['decode']
    sw = r.get('slot_swap') or {}
    print('%-28s b5nr4 %.2f  b5 %.2f  b1nr4 %.2f  BLEU(b5nr4) %.2f  swap %s/%s  n=%s' % (
        r['run'], d['beam5_nr4']['chrF_char'], d['beam5']['chrF_char'], d['beam1_nr4']['chrF_char'],
        d['beam5_nr4']['BLEU'], ('%.3f' % sw['success_swap']) if sw else '-',
        ('%.3f' % sw['success_self']) if sw else '-', sw.get('n_items', '-')))
for f in glob.glob('/kaggle/working/pack/runs'): pass